# Hyperparameter Tuning

We tune the built-in XGBoost estimator here because it emits the `validation:auc` metric
automatically.

Instead of guessing `max_depth`/`eta`/etc., `HyperparameterTuner` runs many training jobs
across a defined search space and picks the best by a chosen objective metric.

> **Note:** the built-in container's metric name uses a colon — `validation:auc` — a
> pre-defined name from AWS's docs. This is different from the `validation-auc` (hyphen)
> string the script-mode notebook prints itself; tuning a script-mode estimator instead
> would need a `metric_definitions` regex to tell the tuner how to parse that custom log
> line, since there's no built-in container emitting a known metric name to key off of.

In [ ]:
import boto3
import pandas as pd
import sagemaker
from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sklearn.model_selection import train_test_split

In [ ]:
# NOTE: Provide the region, bucket name, and execution role ARN for your SageMaker environment
REGION = ""
BUCKET_NAME = ""
EXECUTION_ROLE_ARN = ""

boto_session = boto3.Session(region_name=REGION)
sm_session = sagemaker.Session(boto_session=boto_session)

## Use the data which is already in S3

In [ ]:
# NOTE: Provide the S3 URIs for your training and validation datasets
train_uri = ""
val_uri = ""

# Hardcoding scale_pos_weight from previous knowledge to avoid loading the data
scale_pos_weight = 770

## Base estimator + hyperparameter search space

Only fixed, non-tuned hyperparameters go on the estimator itself (`objective`,
`eval_metric`, `num_round`, `scale_pos_weight`). Everything we want tuned goes in
`hyperparameter_ranges` instead — the tuner overrides these per trial.

* Tunable parameters - `eta`, `max_depth`, `min_child_weight`, `subsample` etc. <br> <br>
* Nontunable parameters - `objective`, `eval_metric`, `num_round`, `scale_pos_weight` <br> <br>

Reference - 
1. https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost-tuning.html
2. https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning-how-it-works.html

In [ ]:
from sagemaker.tuner import ContinuousParameter, HyperparameterTuner, IntegerParameter


# Get the Image URI for xgboost
xgboost_image_uri = image_uris.retrieve("xgboost", REGION, version="1.7-1")

print(f"Image URI : {xgboost_image_uri}")

# 1. Create the estimator
estimator = Estimator(
    image_uri=xgboost_image_uri,
    role=EXECUTION_ROLE_ARN,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{BUCKET_NAME}/models/hpo/",
    sagemaker_session=sm_session,
)
print("Step 1: Estimator created")

# 2. Specify the non tunable parameters
estimator.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    num_round=100,
    scale_pos_weight=scale_pos_weight,
)
print("Step 2: Non tunable hyperparameters set")

# 3. Define the hyperparameter seach space. 
hyperparameter_ranges = {
    "eta": ContinuousParameter(0.01, 0.2),
    "max_depth": IntegerParameter(3, 5),
    "min_child_weight": ContinuousParameter(1, 5),
    "subsample": ContinuousParameter(0.5, 1.0),
}
print("Step 3: Search space for tunable hyperparams defined")

# 4. Create the Hyperparameter Tuner
tuner = HyperparameterTuner(
    estimator=estimator,
    objective_metric_name="validation:auc",
    objective_type="Maximize",
    hyperparameter_ranges=hyperparameter_ranges,
    strategy="Bayesian",
    max_jobs=10,
    max_parallel_jobs=2,
)
print("Step 4: HyperparameterTuner created")

In [ ]:
train_input = TrainingInput(train_uri, content_type="text/csv")
val_input = TrainingInput(val_uri, content_type="text/csv")

tuner.fit({"train": train_input, "validation": val_input})

## Reading tuning results

In [ ]:
results = tuner.analytics().dataframe()
results.sort_values("FinalObjectiveValue", ascending=False).head(10)

In [ ]:
print(f"Best training job: {tuner.best_training_job()}")

In [ ]:
best_estimator = tuner.best_estimator() 
print(f"Best model artifact: {best_estimator.model_data}")

In [ ]:
best_estimator.hyperparameters()